In [3]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from langchain_community.utilities import SQLDatabase
from typing import TypedDict
import os
# DB
db = SQLDatabase.from_uri("sqlite:///sql.db")

api_key = os.getenv('GROQ_API_KEY')
llm = ChatGroq(api_key=api_key,model="llama-3.1-8b-instant")

class State(TypedDict):
    question: str
    query: str
    result: str

def generate_sql(state):
    sql = llm.invoke(f"Write SQL only: {state['question']}").content
    return {"query": sql}

def run_sql(state):
    res = db.run(state["query"])
    return {"result": res}

graph = StateGraph(State)
graph.add_node("sql_gen", generate_sql)
graph.add_node("sql_run", run_sql)

graph.add_edge(START, "sql_gen")
graph.add_edge("sql_gen", "sql_run")
graph.add_edge("sql_run", END)

app = graph.compile()

# Run
app.invoke({"question": "show all employe"})


OperationalError: (sqlite3.OperationalError) near "```sql
SELECT * FROM employees;
```": syntax error
[SQL: ```sql
SELECT * FROM employees;
```]
(Background on this error at: https://sqlalche.me/e/20/e3q8)